In [1]:
## Imports and config
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
from tqdm.auto import tqdm
import copy
import json
import math
import random
import re

/home/erfan/miniconda3/envs/hf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hf_token = json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir = json.load(open(Path("./config.json")))["cache_dir"]

input_path = Path("./dataset_checkpoint.jsonl")
output_path = Path("./data_normal.jsonl")
failure_path = Path("./data_normal_failed_rephrases.jsonl")

target_phrases = [
    "What are the values",
    "Can you provide",
    "I need to find",
]

rephrase_fraction = 0.80
random_seed = 42

In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as fp:
        return [json.loads(line) for line in fp if line.strip()]


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as fp:
        for record in records:
            fp.write(json.dumps(record, ensure_ascii=False) + "\n")


data = read_jsonl(input_path)
len(data)

2100

In [4]:
def select_rephrase_indices(data, target_phrases, fraction=0.80, seed=42):
    rng = random.Random(seed)
    selected_by_phrase = {}

    for phrase in target_phrases:
        matching_indices = [
            idx
            for idx, row in enumerate(data)
            if phrase in row.get("query", "")
        ]
        selected_count = math.floor(len(matching_indices) * fraction)
        selected_indices = sorted(rng.sample(matching_indices, selected_count))

        selected_by_phrase[phrase] = {
            "matched": len(matching_indices),
            "selected": len(selected_indices),
            "indices": selected_indices,
        }

    return selected_by_phrase


selected_by_phrase = select_rephrase_indices(
    data,
    target_phrases,
    fraction=rephrase_fraction,
    seed=random_seed,
)

selected_indices = sorted(
    set().union(*(details["indices"] for details in selected_by_phrase.values()))
)

summary = {
    phrase: {
        "matched": details["matched"],
        "selected_for_rephrase": details["selected"],
    }
    for phrase, details in selected_by_phrase.items()
}
summary["total_unique_selected"] = len(selected_indices)
summary

{'What are the values': {'matched': 280, 'selected_for_rephrase': 224},
 'Can you provide': {'matched': 203, 'selected_for_rephrase': 162},
 'I need to find': {'matched': 474, 'selected_for_rephrase': 379},
 'total_unique_selected': 765}

In [5]:
class QueryRephraser:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=256, temperature=0.7):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p=0.7,
            top_k=10,
            do_sample=True,
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        return self.tokenizer.decode(output_ids, skip_special_tokens=True).strip()


def clean_rephrase(text):
    text = text.strip()
    text = re.sub(r"^```(?:text)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(?:Rephrased query|Rephrase|Query)\s*:\s*", "", text, flags=re.I).strip()
    text = text.strip(' \"\'')
    return " ".join(text.split())


def required_terms(row):
    metadata = row.get("metadata", {})
    terms = []
    terms.extend(metadata.get("companies", []))
    terms.extend(metadata.get("features", []))
    if "start_year" in metadata:
        terms.append(str(metadata["start_year"]))
    if "end_year" in metadata:
        terms.append(str(metadata["end_year"]))
    return [term for term in terms if term]


def preserves_required_terms(row, candidate):
    candidate_lower = candidate.lower()
    return all(str(term).lower() in candidate_lower for term in required_terms(row))


SYSTEM_PROMPT_REPHRASE = """You rewrite financial-data queries. The data set consists of queries about fundamental data of companies.
However these queries are repetetive. So You must change them to normalize the dataset.

Return exactly one rephrased query and nothing else.
In your rephrased query you must use different tone, use of vocabualries and vide variety of words. 
in your outpu tyou MUST not use these terms:
* Can you provide 
* I want to find
* what are the values in your rephrases

Rules:
* Preserve every company name exactly.
* Preserve every metric name exactly.
* Preserve every year exactly.
* Do not add or remove companies, metrics, years, or requested information.
* Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.
"""


def build_rephrase_prompt(row):
    metadata = row.get("metadata", {})
    return f"""Original query:
{row['query']}

Required company names: {metadata.get('companies', [])}
Required metric names: {metadata.get('features', [])}
Required start year: {metadata.get('start_year')}
Required end year: {metadata.get('end_year')}

Rephrased query:"""

In [6]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"

# Use the same Hugging Face loading pattern as data_generation.ipynb.
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto",
)

query_rephraser = QueryRephraser(
    model,
    tokenizer,
    SYSTEM_PROMPT_REPHRASE,
    max_new_tokens=256,
    temperature=0.7,
)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 16735.05it/s]


In [8]:
normalized_data = copy.deepcopy(data)
failed_rephrases = []

for idx in tqdm(selected_indices):
    row = data[idx]
    prompt = build_rephrase_prompt(row)
    original_query = row["query"]
    rephrased_query = None
    candidate = None

    for attempt in range(3):
        candidate = clean_rephrase(query_rephraser.generate(prompt))

        if candidate and candidate != original_query and preserves_required_terms(row, candidate):
            rephrased_query = candidate
            break

    if rephrased_query is None:
        failed_rephrases.append({
            "index": idx,
            "query": original_query,
            "required_terms": required_terms(row),
            "last_candidate": candidate,
        })
        continue

    normalized_data[idx]["orignal_query"] = original_query
    normalized_data[idx]["query"] = rephrased_query

write_jsonl(output_path, normalized_data)
write_jsonl(failure_path, failed_rephrases)

{
    "output_path": str(output_path),
    "failure_path": str(failure_path),
    "selected": len(selected_indices),
    "changed": sum("orignal_query" in row for row in normalized_data),
    "failed": len(failed_rephrases),
}

100%|██████████| 765/765 [2:07:03<00:00,  9.97s/it]  


{'output_path': 'data_normal.jsonl',
 'failure_path': 'data_normal_failed_rephrases.jsonl',
 'selected': 765,
 'changed': 737,
 'failed': 28}